# QuantOne — Kaggle runner (one notebook per model)

Copy this notebook once per model key in `matrix.yaml` and set `MODEL_KEY` below.

**Setup checklist (once per copy):**
1. Settings → Internet → **On** (downloads GGUFs + deps).
2. Accelerator: **GPU T4** for 7–8B models; **CPU** is fine for 1–4B models (those can run as parallel CPU sessions alongside a GPU session).
3. After the first session: **Save Version** so `/kaggle/working/results_<model>.zip` is persisted, then add that notebook's *output* as an **input** of this notebook. Each later session ingests it and resumes — the runner skips every result file that already exists (content-addressed, CLAUDE.md rule 2).
4. Sessions cap ~12h; just re-run the notebook until the final manifest cell prints `TOTAL n/n` complete for this model.

GGUFs are downloaded **one quant at a time and deleted after each quant's run** to stay inside Kaggle's disk budget. sha256 is verified against `matrix.yaml` before every run; a mismatch aborts (rule 5).

In [ ]:
# --- Parameters (edit per copy) ---
MODEL_KEY = "llama32_3b"     # model key from matrix.yaml: llama32_3b | qwen25_3b | gemma2_2b | phi35_mini | smollm2_17b
REPO_GIT_URL = "https://github.com/kaushiksai29/QuantOne.git"  # or attach repo as a dataset and set to None
N_GPU_LAYERS = -1             # -1 = offload all layers; set 0 on CPU sessions
SEEDS = 3                     # matrix.yaml value; do not change mid-study
LIMIT = None                  # None = all 500 tasks; small int for smoke tests

In [ ]:
# --- Get the repo ---
import os, shutil
if REPO_GIT_URL:
    if not os.path.exists("/kaggle/working/QuantOne"):
        !git clone {REPO_GIT_URL} /kaggle/working/QuantOne
else:  # repo attached as a Kaggle dataset named 'quantone-repo'
    shutil.copytree("/kaggle/input/quantone-repo", "/kaggle/working/QuantOne",
                    dirs_exist_ok=True)
%cd /kaggle/working/QuantOne

In [ ]:
# --- Install pinned deps ---
# Prebuilt llama-cpp-python wheels (pinned version) exist for both CPU and CUDA:
#   CPU:  https://abetlen.github.io/llama-cpp-python/whl/cpu
#   CUDA: https://abetlen.github.io/llama-cpp-python/whl/cu124 (T4, CUDA 12.x)
WHEEL_INDEX = "cu124" if N_GPU_LAYERS != 0 else "cpu"
!pip install -q -r requirements.txt --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/{WHEEL_INDEX}
import llama_cpp; print("llama-cpp-python", llama_cpp.__version__)

In [ ]:
# --- Ingest prior results (if any attached) + run this model, all quants ---
import glob
prior = glob.glob("/kaggle/input/*/results_*.zip")
prior_args = " ".join(f"--prior-results {p}" for p in prior)
limit_arg = f"--limit {LIMIT}" if LIMIT else ""
print("prior result zips:", prior or "none")
!python scripts/kaggle_session.py --model {MODEL_KEY} --seeds {SEEDS} \
    {limit_arg} {prior_args} --n-gpu-layers {N_GPU_LAYERS} \
    --zip-to /kaggle/working/results_{MODEL_KEY}.zip

The final manifest above shows per-quant counts. If any cell is short (session timed out), **Save Version**, attach this version's output as an input, and re-run — it resumes where it stopped.